# The personal health dashboard

This notebook is part of the personal health dashboard I am setting up. The goal is to have a centralized place for all my health data to track my progression over time.



In [2]:
import pandas as pd


In [16]:
df = pd.read_csv('../data/hevy/20250426_HevyExport.csv')
df[["start_time", "end_time"]] = df[["start_time", "end_time"]].apply(pd.to_datetime, format="%d %b %Y, %H:%M")
df["date"] = df["start_time"].dt.date
df["id_training"] = df.groupby(["date", "title"]).ngroup()  # Group by workout date and title
df["id_training"] = df["id_training"].astype(str).str.pad(width=3, side="left", fillchar="0")  # Convert to string for unique ID creation

# Group by workout ID
workouts = df.groupby("id_training")
dfs = []  # Create an empty list to store processed DataFrames

df_workouts = []
for id_training, workout in df.groupby("id_training"):
    df_exercises = []
    # Group exercises within each workout and create a unique ID
    ids = workout.groupby("exercise_title").ngroup()  # Group by exercise title
    workout["id_exercise"] = id_training + "_" + ids.astype(str).str.pad(width=2, side="left", fillchar="0")  # Create unique exercise ID
    def one_rep_max(weight, reps):
        return weight * (1 + reps / 30) 
    for id_exercise, exercise in workout.groupby("id_exercise"):
        exercise["one_rep_max"] = max(one_rep_max(exercise["weight_kg"], exercise["reps"]))
        exercise["max_weight"] = exercise["weight_kg"].max()
        df_exercises.append(exercise)
    workout = pd.concat(df_exercises).reset_index(drop=True)  # Concatenate exercises within the workout
    dfs.append(workout)

df_processed = pd.concat(dfs).reset_index(drop=True)  # Concatenate and reset index

df_processed["set_index"] += 1  # Increment set_index to start from 1
df_processed = df_processed[df_processed["reps"].notna()].copy()
df_processed["reps"] = df_processed["reps"].astype(int)  # Convert reps to integer


In [17]:
df_processed

,title,start_time,end_time,description,exercise_title,superset_id,exercise_notes,set_index,set_type,weight_kg,reps,distance_km,duration_seconds,rpe,date,id_training,id_exercise,one_rep_max,max_weight
0,Push,2022-06-06 13:58:00,2022-06-06 15:05:00,"Last van onderrug, eig moest ik benen",Bench Press (Barbell),NaN,NaN,1,normal,20.0,20,NaN,NaN,NaN,2022-06-06,000,000_00,96.000000,80.0
1,Push,2022-06-06 13:58:00,2022-06-06 15:05:00,"Last van onderrug, eig moest ik benen",Bench Press (Barbell),NaN,NaN,2,normal,60.0,10,NaN,NaN,NaN,2022-06-06,000,000_00,96.000000,80.0
2,Push,2022-06-06 13:58:00,2022-06-06 15:05:00,"Last van onderrug, eig moest ik benen",Bench Press (Barbell),NaN,NaN,3,normal,80.0,6,NaN,NaN,NaN,2022-06-06,000,000_00,96.000000,80.0
3,Push,2022-06-06 13:58:00,2022-06-06 15:05:00,"Last van onderrug, eig moest ik benen",Bench Press (Barbell),NaN,NaN,4,normal,70.0,8,NaN,NaN,NaN,2022-06-06,000,000_00,96.000000,80.0
4,Push,2022-06-06 13:58:00,2022-06-06 15:05:00,"Last van onderrug, eig moest ik benen",Bench Press (Barbell),NaN,NaN,5,normal,60.0,10,NaN,NaN,NaN,2022-06-06,000,000_00,96.000000,80.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3732,Afternoon workout 💪,2025-04-14 16:28:00,2025-04-14 17:20:00,NaN,Shoulder Press (Dumbbell),NaN,NaN,2,normal,10.0,11,NaN,NaN,NaN,2025-04-14,196,196_04,13.666667,10.0
3733,Afternoon workout 💪,2025-04-14 16:28:00,2025-04-14 17:20:00,NaN,Shoulder Press (Dumbbell),NaN,NaN,3,normal,10.0,9,NaN,NaN,NaN,2025-04-14,196,196_04,13.666667,10.0
3734,Afternoon workout 💪,2025-04-14 16:28:00,2025-04-14 17:20:00,NaN,Triceps Rope Pushdown,NaN,NaN,1,normal,12.5,14,NaN,NaN,NaN,2025-04-14,196,196_05,18.500000,15.0
3735,Afternoon workout 💪,2025-04-14 16:28:00,2025-04-14 17:20:00,NaN,Triceps Rope Pushdown,NaN,NaN,2,normal,15.0,7,NaN,NaN,NaN,2025-04-14,196,196_05,18.500000,15.0


In [7]:
df_hevy.head()

,title,start_time,end_time,description,exercise_title,superset_id,exercise_notes,set_index,set_type,weight_kg,reps,distance_km,duration_seconds,rpe
0,Afternoon workout 💪,"14 Apr 2025, 16:28","14 Apr 2025, 17:20",NaN,Incline Bench Press (Barbell),NaN,NaN,0,normal,20.0,15.0,NaN,NaN,NaN
1,Afternoon workout 💪,"14 Apr 2025, 16:28","14 Apr 2025, 17:20",NaN,Incline Bench Press (Barbell),NaN,NaN,1,normal,30.0,12.0,NaN,NaN,NaN
2,Afternoon workout 💪,"14 Apr 2025, 16:28","14 Apr 2025, 17:20",NaN,Incline Bench Press (Barbell),NaN,NaN,2,normal,35.0,10.0,NaN,NaN,NaN
3,Afternoon workout 💪,"14 Apr 2025, 16:28","14 Apr 2025, 17:20",NaN,Incline Bench Press (Barbell),NaN,NaN,3,normal,35.0,10.0,NaN,NaN,NaN
4,Afternoon workout 💪,"14 Apr 2025, 16:28","14 Apr 2025, 17:20",NaN,Bench Press (Dumbbell),NaN,NaN,0,normal,12.0,12.0,NaN,NaN,NaN
